In [16]:
# ==============================================================================
# SCRIPT:         colab_keystone_generator_v6.2_definitive.ipynb
# OPERATION:      Operation Keystone (Definitive Build)
# PURPOSE:        The final, correct script. It uses a safe, non-destructive
#                 method to build the final DataFrame, guaranteeing both unique
#                 Master_IDs and the integrity of the original data structure.
# ==============================================================================

# --- 1. Setup ---
!pip install -q gspread
import pandas as pd
import hashlib
import os
from google.colab import auth, drive
import gspread
from google.auth import default
import numpy as np

print("Authenticating...")
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
drive.mount('/content/drive', force_remount=True)
print("✅ Authentication and Drive Mount successful.")

# --- 2. CONFIGURATION ---
SPREADSHEET_NAME_INPUT = 'raw_Consomaster'
WORKSHEET_NAME_INPUT = 'consomaster'
OUTPUT_FOLDER_PATH = '/content/drive/My Drive/Opus_1_101001/Database/03_analysis/'
OUTPUT_FILENAME = 'ConsoMaster_with_MasterID_FINAL.csv'

# --- DEFINITIVE Master_ID Generation Function (Confirmed Correct) ---
def generate_master_id(row):
    """Constructs a signature string using positional iloc and robust checks."""
    proxy_id = str(row.iloc[0]) if pd.notna(row.iloc[0]) else f"row_{row.name}"
    uber_ts, gts_ts = row.iloc[7], row.iloc[3]
    raw_date, raw_time = row.iloc[16], row.iloc[18]
    uber_status, raw_outcome = row.iloc[10], row.iloc[22]
    event_id = str(row.iloc[27]) if pd.notna(row.iloc[27]) else "no_event"

    ts_str = ''
    if pd.notna(uber_ts): ts_str = str(pd.to_datetime(uber_ts, errors='coerce'))
    elif pd.notna(gts_ts): ts_str = str(pd.to_datetime(gts_ts, errors='coerce'))
    elif pd.notna(raw_date): ts_str = str(raw_date) + " " + str(raw_time)
    else: ts_str = "no_timestamp"

    status = str(uber_status) if pd.notna(uber_status) else str(raw_outcome) if pd.notna(raw_outcome) else 'unknown_status'

    signature_str = f"{proxy_id}::{ts_str}::{status}::{event_id}".lower().encode('utf-8')
    return hashlib.sha256(signature_str).hexdigest()

try:
    # --- 3. Load Data Components ---
    print(f"\nLoading data from sheet '{WORKSHEET_NAME_INPUT}'...")
    worksheet = gc.open(SPREADSHEET_NAME_INPUT).worksheet(WORKSHEET_NAME_INPUT)
    all_values = worksheet.get_all_values()

    # Separate headers from data rows
    original_headers = all_values[0]
    data_rows = all_values[1:]

    # Load data into a temporary, headless DataFrame for processing
    df_temp = pd.DataFrame(data_rows)
    df_temp.replace('', np.nan, inplace=True)
    print(f"Loaded {len(df_temp)} records.")

    # --- 4. Generate Master_ID ---
    print("Applying final Master_ID generation protocol...")
    master_id_series = df_temp.apply(generate_master_id, axis=1)

    # --- 5. SAFE FINALIZATION: Build a New, Correct DataFrame ---
    print("Constructing the final, correct data structure...")
    # First, create the output DataFrame with the original data and original headers.
    df_final = pd.DataFrame(data_rows, columns=original_headers)

    # Now, use the safe '.insert()' method to add the Master_ID as the very first column.
    # This physically adds the new column and shifts everything else correctly.
    df_final.insert(0, 'Master_ID', master_id_series)

    # --- 6. Verification and Save ---
    is_unique = df_final['Master_ID'].nunique() == len(df_final)
    print("  > Master_ID generation and final construction complete.")
    if is_unique:
        print("  > ✅ VERIFICATION PASSED: All generated Master_IDs are unique.")
    else:
        duplicate_count = len(df_final) - df_final['Master_ID'].nunique()
        print(f"  > ❌ VERIFICATION FAILED: Found {duplicate_count} duplicate Master_IDs. Critical error.")

    full_output_path = os.path.join(OUTPUT_FOLDER_PATH, OUTPUT_FILENAME)
    df_final.to_csv(full_output_path, index=False)

    print(f"\n✅ Mission Success. Data with definitive Master_IDs saved to a CSV file at:")
    print(f"   > '{full_output_path}'")

except Exception as e:
    print(f"❌ An unrecoverable error occurred: {e}")

Authenticating...
Mounted at /content/drive
✅ Authentication and Drive Mount successful.

Loading data from sheet 'consomaster'...
Loaded 353 records.
Applying final Master_ID generation protocol...
Constructing the final, correct data structure...
  > Master_ID generation and final construction complete.
  > ✅ VERIFICATION PASSED: All generated Master_IDs are unique.

✅ Mission Success. Data with definitive Master_IDs saved to a CSV file at:
   > '/content/drive/My Drive/Opus_1_101001/Database/03_analysis/ConsoMaster_with_MasterID_FINAL.csv'


/tmp/ipython-input-1247688718.py:64: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_temp.replace('', np.nan, inplace=True)
